# 03 – Visualization Tour

This notebook demonstrates AMSA's visualization capabilities by rendering the examples from the `/examples` folder.
We cover 2D trajectories, geometric constructions, and spatial relationships.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from amsa import Algebra
from amsa.viz.adapters import to_point
from amsa.viz.backends import mpl

## 3.1 Circular Robot Motion (PGA2d)

A differential-drive robot moving forward while turning at a constant rate follows a circular trajectory.
We use a motor (translator × rotor) and apply it via sandwich conjugation.

In [ ]:
alg = Algebra.pga2d()

theta = np.deg2rad(10)
forward_step = 0.5
steps = 36

# Rotor (rotation)
rotor = alg.multivector({
    "e": np.cos(theta / 2),
    "e12": -np.sin(theta / 2)
}).normalized()

# Translator
translator = alg.multivector({
    "e": 1.0,
    "e02": 0.5 * forward_step,
})

# Motor = translator * rotor
motor = translator * rotor

# Starting point
point = alg.multivector({
    "e01": 0.0,
    "e02": 0.0,
    "e12": 1.0,
})

# Compute trajectory
trajectory = []
for i in range(steps):
    point = motor.sandwich(point)
    trajectory.append([point.component("e01"), point.component("e02")])

trajectory = np.array(trajectory)

# Visualize
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(trajectory[:, 0], trajectory[:, 1], 'b-', linewidth=1.5, label='Trajectory')
ax.scatter(trajectory[0, 0], trajectory[0, 1], c='green', s=100, zorder=5, label='Start')
ax.scatter(trajectory[-1, 0], trajectory[-1, 1], c='red', s=100, zorder=5, label='End')
ax.scatter(0, 0, c='black', s=50, marker='x', zorder=5)
ax.set_aspect('equal', 'box')
ax.set_title('Circular Robot Motion (PGA2d Motor)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 3.2 Rigid Body Trajectory (PGA2d)

A motor combining rotation and translation produces a more general rigid-body motion.

In [ ]:
alg = Algebra.pga2d()

theta = np.deg2rad(10)
tx, ty = 0.5, 0.0
steps = 20

# Rotor
rotor = alg.multivector({
    "e": np.cos(theta / 2),
    "e12": -np.sin(theta / 2)
}).normalized()

# Translator
translator = alg.multivector({
    "e": 1.0,
    "e01": -0.5 * ty,
    "e02": 0.5 * tx,
})

motor = translator * rotor

# Starting point
point = alg.multivector({
    "e01": 0.0,
    "e02": 0.0,
    "e12": 1.0,
})

# Compute trajectory
trajectory = []
for i in range(steps):
    point = motor.sandwich(point)
    trajectory.append([point.component("e01"), point.component("e02")])

trajectory = np.array(trajectory)

# Visualize
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(trajectory[:, 0], trajectory[:, 1], 'b-o', linewidth=1, markersize=4, label='Trajectory')
ax.scatter(trajectory[0, 0], trajectory[0, 1], c='green', s=100, zorder=5, label='Start')
ax.scatter(trajectory[-1, 0], trajectory[-1, 1], c='red', s=100, zorder=5, label='End')
ax.set_aspect('equal', 'box')
ax.set_title('Rigid Body Trajectory (PGA2d)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 3.3 Triangle Area (VGA2d)

The wedge product of two edge vectors produces an oriented area bivector.
We visualize three cases: right triangle, flipped orientation, and skew triangle.

In [ ]:
alg = Algebra.vga2d()

# Right triangle
p = alg.vector([0.0, 0.0])
q = alg.vector([5.0, 0.0])
r = alg.vector([0.0, 3.0])

u = q - p
v = r - p
area_bivector = u ^ v
area = area_bivector.component("e12") / 2.0

# Visualize
fig, ax = plt.subplots(figsize=(6, 6))

pts = np.array([[0, 0], [5, 0], [0, 3], [0, 0]])
ax.plot(pts[:, 0], pts[:, 1], 'b-', linewidth=2)
ax.scatter(pts[:, 0], pts[:, 1], c=['red', 'blue', 'blue'], s=100, zorder=5)
ax.text(0.15, 0.15, 'p', fontsize=12)
ax.text(5.1, 0, 'q', fontsize=12)
ax.text(0.1, 3.1, 'r', fontsize=12)

# Draw edge vectors
ax.annotate('', xy=(5, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='green', lw=1.5))
ax.annotate('', xy=(0, 3), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='purple', lw=1.5))
ax.text(2.5, -0.3, 'u = q-p', fontsize=10, color='green')
ax.text(0.4, 1.5, 'v = r-p', fontsize=10, color='purple')

ax.set_xlim(-0.5, 6)
ax.set_ylim(-0.5, 4)
ax.set_aspect('equal', 'box')
ax.set_title(f'Triangle Area = {area:.2f} (CCW orientation)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
plt.show()

print(f"Signed area (right triangle): {area}")

In [ ]:
# Flipped orientation
flipped_area_bivector = v ^ u
flipped_area = flipped_area_bivector.component("e12") / 2.0
print(f"Flipped orientation area: {flipped_area}")

# Skew triangle
p = alg.vector([1.5, 2.0])
q = alg.vector([4.0, 3.0])
r = alg.vector([2.0, 7.0])
u = q - p
v = r - p
skew_area_bivector = u ^ v
skew_area = skew_area_bivector.component("e12") / 2.0
print(f"Skew triangle area: {skew_area}")

## 3.4 Planar Heading Update (VGA2d)

A rotor rotates a vector via sandwich conjugation. We visualize the robot's forward axis before and after rotation.

In [ ]:
alg = Algebra.vga2d()

forward_body = alg.vector([1.0, 0.0])

theta = np.deg2rad(30)
rotor = alg.multivector({"e": np.cos(theta / 2), "e12": -np.sin(theta / 2)}).normalized()

forward_world = rotor.sandwich(forward_body)

# Visualize
fig, ax = plt.subplots(figsize=(6, 6))

# Original direction (body frame)
ax.annotate('', xy=(1.5, 0), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.text(1.1, -0.2, 'Body forward', fontsize=10, color='blue')

# Rotated direction (world frame)
fw = forward_world.grade(1).values
ax.annotate('', xy=(fw[0]*1.5, fw[1]*1.5), xytext=(0, 0), arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.text(fw[0]*1.6, fw[1]*1.6+0.1, 'World forward', fontsize=10, color='red')

# Draw rotation arc
arc = plt.Circle((0, 0), 0.8, fill=False, color='gray', linestyle='--')
ax.add_patch(arc)

ax.set_xlim(-0.5, 2)
ax.set_ylim(-0.5, 1)
ax.set_aspect('equal', 'box')
ax.set_title(f'Heading Update: {np.rad2deg(theta)}° rotation')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
plt.show()

## 3.5 Plane Reflections (VGA3d)

Reflecting a vector across a plane uses the sandwich product with the plane normal.
We visualize a velocity vector reflected across the three coordinate planes.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

alg = Algebra.vga3d()

velocity = alg.vector([1.0, -2.0, 0.5])

nx = alg.vector([1.0, 0.0, 0.0]).normalized()  # YZ plane
ny = alg.vector([0.0, 1.0, 0.0]).normalized()  # XZ plane
nz = alg.vector([0.0, 0.0, 1.0]).normalized()  # XY plane

reflect_x = -nx.sandwich(velocity)
reflect_y = -ny.sandwich(velocity)
reflect_z = -nz.sandwich(velocity)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

orig = velocity.grade(1).values
rx = reflect_x.grade(1).values
ry = reflect_y.grade(1).values
rz = reflect_z.grade(1).values

origin = [0, 0, 0]

# Original
ax.quiver(*origin, *orig, color='black', linewidth=2, label='Original')

# Reflections
ax.quiver(*origin, *rx, color='red', linewidth=1.5, label='YZ reflection')
ax.quiver(*origin, *ry, color='green', linewidth=1.5, label='XZ reflection')
ax.quiver(*origin, *rz, color='blue', linewidth=1.5, label='XY reflection')

# Planes (as lines at origin)
ax.plot([0, 2], [0, 0], [0, 0], 'r--', alpha=0.3)  # x-axis
ax.plot([0, 0], [0, -3], [0, 0], 'g--', alpha=0.3)  # y-axis
ax.plot([0, 0], [0, 0], [0, 1], 'b--', alpha=0.3)  # z-axis

ax.set_xlim(-2, 2)
ax.set_ylim(-3, 1)
ax.set_zlim(-1, 2)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Plane Reflections')
ax.legend()
plt.show()

## 3.6 Point to Plane Distance (VGA3d)

A plane is represented by its spanning vectors. The dual gives the normal.
We compute and visualize the signed distance from a point to the plane.

In [ ]:
alg = Algebra.vga3d()

# Plane origin
p0 = alg.vector([0.0, 0.0, 0.0])

# Plane spanning vectors
u = alg.vector([1.0, 0.0, 0.0])
v = alg.vector([0.0, 1.0, 0.0])

# Plane bivector
B = u ^ v
n = B.dual()

# Point above the plane
p = alg.vector([0.5, 0.5, 2.0])

# Displacement
d = p - p0

# Signed distance
distance = (d | n).component("e") / np.linalg.norm(n.values)

print(f"Signed distance to plane: {distance}")

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Plane (XY plane at z=0)
xx, yy = np.meshgrid(np.linspace(-1, 1, 5), np.linspace(-1, 1, 5))
zz = np.zeros_like(xx)
ax.plot_surface(xx, yy, zz, alpha=0.3, color='cyan')

# Point
px, py, pz = 0.5, 0.5, 2.0
ax.scatter([px], [py], [pz], c='red', s=100, label=f'Point ({px},{py},{pz})')

# Normal vector
nx_vec = n.grade(1).values
ax.quiver(0, 0, 0, nx_vec[0], nx_vec[1], nx_vec[2], color='green', linewidth=2, label='Normal')

# Perpendicular line to plane
ax.plot([px, px], [py, py], [0, pz], 'r--', linewidth=1, alpha=0.7)

ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(-0.5, 2.5)
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title(f'Point to Plane Distance = {distance:.2f}')
ax.legend()
plt.show()

## 3.7 Trilateration (PGA2d)

Robot localization by measuring distances to three beacons.

In [ ]:
alg = Algebra.pga2d()

# Beacon locations
b1 = alg.multivector({"e01": 0.0, "e02": 0.0, "e12": 1.0})
b2 = alg.multivector({"e01": 6.0, "e02": 0.0, "e12": 1.0})
b3 = alg.multivector({"e01": 3.0, "e02": 5.0, "e12": 1.0})

# Robot location
robot = alg.multivector({"e01": 3.0, "e02": 2.0, "e12": 1.0})

# Compute distances
def dist(p, q):
    dx = p.component("e01") - q.component("e01")
    dy = p.component("e02") - q.component("e02")
    return np.sqrt(dx * dx + dy * dy)

d1 = dist(robot, b1)
d2 = dist(robot, b2)
d3 = dist(robot, b3)

# Visualize
fig, ax = plt.subplots(figsize=(6, 6))

beacons = [b1, b2, b3]
distances = [d1, d2, d3]
colors = ['blue', 'blue', 'blue']

for i, (b, d, c) in enumerate(zip(beacons, distances, colors)):
    pt = to_point(b)
    ax.scatter(pt.position[0], pt.position[1], c=c, s=100, zorder=5)
    circle = plt.Circle(pt.position, d, color=c, fill=False, linestyle='--', alpha=0.5)
    ax.add_patch(circle)
    ax.text(pt.position[0]+0.2, pt.position[1]+0.2, f'B{i+1}', fontsize=10)

# Robot
robot_pt = to_point(robot)
ax.scatter(robot_pt.position[0], robot_pt.position[1], c='red', s=150, marker='*', zorder=6, label='Robot')

ax.set_aspect('equal', 'box')
ax.set_title('Robot Trilateration')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

print(f"Distances: B1={d1:.2f}, B2={d2:.2f}, B3={d3:.2f}")

## Summary

This notebook demonstrated visualization of:

- **PGA2d trajectories**: Circular and rigid-body motion via motors
- **VGA2d geometry**: Triangle area, heading updates with rotors
- **VGA3d geometry**: Plane reflections, signed volume, point-plane distance
- **PGA2d localization**: Trilateration with distance circles

The `amsa.viz` module provides neutral primitives (`Point`, `Line`, `Plane`) and backends (matplotlib) for rendering multivector data.